# Synthetic EHR Data Generation

In [1]:
import numpy as np
import pandas as pd
from faker import Faker
from datetime import timedelta

In [2]:
fake=Faker('en_IN')
np.random.seed(42)
n_admissions = 1000000
n_patients = 775600

## Table 1 : Patient Demographics

In [3]:
patients = pd.DataFrame({
    'patient_id':[f'P{i:05d}' for i in range(n_patients)],
    'age':np.random.randint(18,90,n_patients),
    'gender':np.random.choice(['M','F','Other'],n_patients,p=[0.48,0.50,0.02]),
    'blood_group':np.random.choice(['A+','B+','O+','AB+','A-','B-','O-','AB-'],n_patients),
    'insurance_type':np.random.choice(['government','private','self_pay'],n_patients,p=[0.35,0.40,0.25]),
    'district':np.random.choice(['urban','semi_urban','rural'],n_patients,p=[0.4,0.35,0.25]),
    'chronic_conditions':np.random.choice([0,1,2,3,4],n_patients,p=[0.25,0.30,0.25,0.15,0.05]),
})

## Table 2 : Admissions

In [4]:
admissions=pd.DataFrame({
    'admission_id':[f'A{i:06d}' for i in range(n_admissions)],
    'patient_id':np.random.choice(patients['patient_id'],n_admissions),
    'admission_date':[fake.date_between(start_date='-2y',end_date='today') for _ in range (n_admissions)],
    'department':np.random.choice(['cardiology','pulmonology','endocrinology','orthopedics','general_medicine','neurology'],n_admissions,p=[0.20,0.15,0.15,0.15,0.25,0.10]),
    'admission_type':np.random.choice(['emergency','elective','urgent'],n_admissions,p=[0.40,0.35,0.25]),
    'length_of_stay_days':np.random.lognormal(1.2,0.7,n_admissions).clip(1,45).astype(int),
    'num_procedures':np.random.poisson(2,n_admissions),
    'num_medications_discharge':np.random.randint(1,12,n_admissions),
    'discharge_disposition':np.random.choice(['home','home_health_service','rehab','skilled_nursing'],n_admissions,p=[0.60,0.20,0.10,0.10]),
})

## Table 3 : Labs

In [5]:
labs = pd.DataFrame({
    'admission_id':admissions['admission_id'],
    'hemoglobin':np.round(np.random.normal(12.5,2.5,n_admissions).clip(5,20),1),
    'wbc_count':np.round(np.random.lognormal(2.1,0.4,n_admissions).clip(2,30),1),
    'creatinine':np.round(np.random.lognormal(0.1,0.5,n_admissions).clip(0.3,12),2),
    'glucose_random':np.round(np.random.normal(140,50,n_admissions).clip(50,500),0),
    'sodium':np.round(np.random.normal(140,4,n_admissions).clip(120,160),0),
    'potassium':np.round(np.random.normal(4.2,0.6,n_admissions).clip(2.5,7),1),
})

### Introducing Missing Value 

In [6]:
# Adding ~8% missing value to the labs table
for col in ['hemoglobin','creatinine','glucose_random']:
    labs.loc[np.random.random(n_admissions) < 0.08,col] = np.nan

## Export to csv files

In [7]:
patients.to_csv('data/patients.csv',index=False)
admissions.to_csv('data/admissions.csv',index=False)
labs.to_csv('data/labs.csv',index=False)